# TransNet Reproduction on HAM10000

This notebook reproduces the paper **"Skin Lesion Classification Using a Hybrid ResNet–Transformer Encoder Architecture"** (Shah & Khan, ICIC 2025) on the HAM10000 dataset.
It follows the exact split CSVs provided in the original notebook (train/val/test).
You can run the dry‑run first (2 epochs) and then the full training (42 epochs).

In [ ]:
# 1️⃣ Mount Google Drive (required for data access)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2️⃣ Set project directory on Drive
project_dir = '/content/drive/MyDrive/HAM10000_transnet'
!mkdir -p "$project_dir"
%cd $project_dir

In [ ]:
# 3️⃣ Install required libraries (GPU‑enabled PyTorch is pre‑installed)
!pip install numpy pandas scikit-learn pillow matplotlib --quiet

In [ ]:
# 4️⃣ Configuration (paper §IV.C)
import os, json, warnings, glob, random
import numpy as np, pandas as pd
import torch, torch.nn as nn
import torchvision, torchvision.transforms as transforms, torchvision.models as models
from torchvision.transforms import InterpolationMode
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score, precision_score, recall_score, roc_auc_score, confusion_matrix)
warnings.filterwarnings('ignore', category=UserWarning)

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = device.type == 'cuda'
print(f'Device: {device} | Mixed‑precision: {USE_AMP}')

# Dataset parameters
DATA_DIR = '/content/drive/MyDrive/HAM10000'  # adjust if needed
SPLIT_DIR = DATA_DIR
IMG_SIZE = 224
NUM_CLASSES = 7
BATCH_SIZE = 32  # assumed
EPOCHS = 42
DRY_RUN = False

# Augmentation parameters (paper §III.B)
AUG_ROT = 30
AUG_FLIP = 0.5
AUG_SCALE = (0.8, 1.2)
AUG_CROP_FROM = 256
AUG_BRIGHT = 0.2
AUG_CONTRAST = 0.15
AUG_SAT = 0.1
AUG_HUE = 0.02
AUG_NOISE = 0.01

# Model hyper‑parameters (paper §III.D)
D_MODEL = 96
NHEAD = 8
NUM_LAYERS = 6
DIM_FFN = 384
DROPOUT = 0.1

# Optimizer & scheduler (paper §IV.C)
LR = 1e-4
WEIGHT_DECAY = 1e-4
BETAS = (0.9, 0.999)
EPS = 1e-8
GRAD_CLIP = 1.0
LR_FACTOR = 0.5
LR_PATIENCE = 5
LR_MIN = 1e-7

# Balancing target (paper §III.A.3)
BALANCE_TARGET = 500


In [ ]:
# 5️⃣ Helper functions
def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def build_transforms():
    mean = [0.485, 0.456, 0.406]
    std  = [0.229, 0.224, 0.225]
    train_tf = transforms.Compose([
        transforms.Resize((AUG_CROP_FROM, AUG_CROP_FROM)),
        transforms.RandomHorizontalFlip(p=AUG_FLIP),
        transforms.RandomVerticalFlip(p=AUG_FLIP),
        transforms.RandomAffine(degrees=AUG_ROT, scale=AUG_SCALE),
        transforms.RandomCrop(IMG_SIZE),
        transforms.ColorJitter(brightness=AUG_BRIGHT, contrast=AUG_CONTRAST,
                               saturation=AUG_SAT, hue=AUG_HUE),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
        AddGaussianNoise(std=AUG_NOISE)
    ])
    eval_tf = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE), interpolation=InterpolationMode.BICUBIC),
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ])
    return train_tf, eval_tf

class AddGaussianNoise:
    def __init__(self, std=AUG_NOISE):
        self.std = std
    def __call__(self, tensor):
        return tensor + torch.randn_like(tensor) * self.std
    def __repr__(self):
        return f'AddGaussianNoise(std={self.std})'

In [ ]:
# 6️⃣ Data loading (uses the CSV splits)
def load_data():
    meta = pd.read_csv(os.path.join(DATA_DIR, 'HAM10000_metadata.csv'))
    train_df = pd.read_csv(os.path.join(SPLIT_DIR, 'train_split.csv'))
    val_df   = pd.read_csv(os.path.join(SPLIT_DIR, 'val_split.csv'))
    test_df  = pd.read_csv(os.path.join(SPLIT_DIR, 'test_split.csv'))
    img_dirs = [os.path.join(DATA_DIR, 'HAM10000_images_part_1'),
                os.path.join(DATA_DIR, 'HAM10000_images_part_2')]
    img_map = {}
    for d in img_dirs:
        if not os.path.isdir(d):
            continue
        for p in glob.glob(os.path.join(d, '*.jpg')):
            img_map[os.path.splitext(os.path.basename(p))[0]] = p
    classes = sorted(meta['dx'].unique())
    class2idx = {c:i for i,c in enumerate(classes)}
    for df in (train_df, val_df, test_df):
        df['image_path'] = df['image_id'].map(img_map)
        df['label'] = df['dx'].map(class2idx)
    return train_df, val_df, test_df, classes, class2idx

In [ ]:
# 7️⃣ Balancing (paper §III.A.3)
def balance_dataset(df, classes, class2idx, target=BALANCE_TARGET, seed=SEED):
    rng = np.random.RandomState(seed)
    parts = []
    for cls in classes:
        idx = class2idx[cls]
        sub = df[df['label'] == idx].copy()
        n = len(sub)
        if n > target:
            sub = sub.sample(target, random_state=seed, replace=False)
        elif n < target:
            extra = sub.sample(target - n, random_state=seed, replace=True)
            sub = pd.concat([sub, extra], ignore_index=True)
        parts.append(sub)
    balanced = pd.concat(parts, ignore_index=True).sample(frac=1, random_state=seed).reset_index(drop=True)
    return balanced

In [ ]:
# 8️⃣ Dataset class
class HAM10000Dataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['image_path']).convert('RGB')
        label = int(row['label'])
        if self.transform:
            img = self.transform(img)
        return img, label

In [ ]:
# 9️⃣ TransNet model (paper §III.D)
class TransNet(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, d_model=D_MODEL, nhead=NHEAD,
                 num_layers=NUM_LAYERS, dim_feedforward=DIM_FFN, dropout=DROPOUT, pretrained=True):
        super().__init__()
        resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT if pretrained else None)
        self.backbone = nn.Sequential(*list(resnet.children())[:-2])  # remove avg‑pool & fc
        self.projection = nn.Linear(512, d_model)
        self.pos_encoding = nn.Parameter(torch.randn(1, 49, d_model) * 0.02)
        enc_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead,
                                            dim_feedforward=dim_feedforward,
                                            dropout=dropout, activation='gelu',
                                            norm_first=True, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.classifier = nn.Linear(d_model, num_classes)
        self._init_weights()
    def _init_weights(self):
        nn.init.xavier_uniform_(self.projection.weight)
        nn.init.zeros_(self.projection.bias)
        nn.init.xavier_uniform_(self.classifier.weight)
        nn.init.zeros_(self.classifier.bias)
    def forward(self, x):
        feats = self.backbone(x)  # [B,512,7,7]
        B, C, H, W = feats.shape
        tokens = feats.flatten(2).transpose(1,2)  # [B,49,512]
        tokens = self.projection(tokens)
        tokens = tokens + self.pos_encoding
        tokens = self.transformer_encoder(tokens)
        pooled = tokens.mean(dim=1)
        logits = self.classifier(pooled)
        return logits

In [ ]:
# 🔟 Optimizer, scheduler & loss
def build_optimizer_and_scheduler(model):
    decay, no_decay = [], []
    for n, p in model.named_parameters():
        if not p.requires_grad: continue
        if 'bn' in n or 'norm' in n:
            no_decay.append(p)
        else:
            decay.append(p)
    optimizer = torch.optim.Adam([{'params': decay, 'weight_decay': WEIGHT_DECAY},
                                     {'params': no_decay, 'weight_decay': 0.0}],
                                    lr=LR, betas=BETAS, eps=EPS)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=LR_FACTOR, patience=LR_PATIENCE, min_lr=LR_MIN)
    return optimizer, scheduler

criterion = nn.CrossEntropyLoss()

In [ ]:
# 1️⃣1️⃣ Evaluation utilities
def evaluate(loader, model):
    model.eval()
    total_loss, total_n = 0.0, 0
    all_logits, all_labels = [], []
    with torch.no_grad():
        for imgs, lbls in loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            logits = model(imgs)
            loss = criterion(logits, lbls)
            total_loss += loss.item() * lbls.size(0)
            total_n += lbls.size(0)
            all_logits.append(logits.cpu())
            all_labels.append(lbls.cpu())
    avg_loss = total_loss / total_n
    return avg_loss, torch.cat(all_logits), torch.cat(all_labels)

def compute_metrics(logits, labels, class_names):
    preds = logits.argmax(dim=1).numpy()
    y = labels.numpy()
    probs = torch.softmax(logits, dim=1).numpy()
    res = {
        'accuracy': accuracy_score(y, preds),
        'macro_f1': f1_score(y, preds, average='macro'),
        'per_class': {
            cls: {
                'precision': precision_score(y, preds, labels=[i], average='macro'),
                'recall': recall_score(y, preds, labels=[i], average='macro'),
                'f1': f1_score(y, preds, labels=[i], average='macro')
            } for i,cls in enumerate(class_names)
        }
    }
    return res

In [ ]:
# 1️⃣2️⃣ Main training loop
def train(model, train_loader, val_loader, optimizer, scheduler, epochs):
    scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)
    best_val_loss = float('inf')
    best_state = None
    history = []
    for epoch in range(1, epochs+1):
        model.train()
        train_loss = 0.0
        n = 0
        for imgs, lbls in train_loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
                logits = model(imgs)
                loss = criterion(logits, lbls)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
            train_loss += loss.item() * lbls.size(0)
            n += lbls.size(0)
        train_loss /= n
        val_loss, val_logits, val_labels = evaluate(val_loader, model)
        scheduler.step(val_loss)
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k:v.cpu().clone() for k,v in model.state_dict().items()}
        history.append({'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss,
                        'lr': optimizer.param_groups[0]['lr']})
        print(f"Epoch {epoch:02d}/{epochs} – train {train_loss:.4f} – val {val_loss:.4f} – lr {optimizer.param_groups[0]['lr']:.1e}")
    return best_state, history

In [ ]:
# 1️⃣3️⃣ Run training / evaluation
def main():
    os.makedirs('output', exist_ok=True)
    train_df, val_df, test_df, classes, class2idx = load_data()
    train_bal = balance_dataset(train_df, classes, class2idx)
    if DRY_RUN:
        train_bal = train_bal.groupby('label').head(30).reset_index(drop=True)
        val_df = val_df.groupby('label').head(10).reset_index(drop=True)
        test_df = test_df.groupby('label').head(10).reset_index(drop=True)
    train_tf, eval_tf = build_transforms()
    train_set = HAM10000Dataset(train_bal, train_tf)
    val_set   = HAM10000Dataset(val_df, eval_tf)
    test_set  = HAM10000Dataset(test_df, eval_tf)
    generator = torch.Generator().manual_seed(SEED)
    train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=2, pin_memory=device.type=='cuda', generator=generator)
    val_loader   = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=2, pin_memory=device.type=='cuda', generator=generator)
    test_loader  = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=2, pin_memory=device.type=='cuda', generator=generator)
    model = TransNet(pretrained=True).to(device)
    optimizer, scheduler = build_optimizer_and_scheduler(model)
    epochs = 2 if DRY_RUN else EPOCHS
    best_state, history = train(model, train_loader, val_loader, optimizer, scheduler, epochs)
    if best_state is not None:
        model.load_state_dict(best_state)
    test_loss, test_logits, test_labels = evaluate(test_loader, model)
    metrics = compute_metrics(test_logits, test_labels, classes)
    result = {
        'dry_run': DRY_RUN,
        'epochs_trained': epochs,
        'test_metrics': metrics,
        'history': history
    }
    result_path = os.path.join('output', f"results_{'DRY' if DRY_RUN else 'FULL'}.json")
    with open(result_path, 'w') as f:
        json.dump(result, f, indent=2)
    print('✅ Results saved to', os.path.abspath(f.name))

if __name__ == '__main__':
    main()

In [ ]:
# 1️⃣4️⃣ Quick dry‑run (2 epochs) – comment out to skip
DRY_RUN = True
!python train_transnet.py --dry-run  # uses the same script defined above

In [ ]:
# 1️⃣5️⃣ Full training (42 epochs) – uncomment when ready
# DRY_RUN = False
# !python train_transnet.py